# P73 — Cuantización por mínimos cuadrados en PCM

## 1. Título y paper

**Paper:** *Least Squares Quantization in PCM*  
**Autoría:** Stuart P. Lloyd  
**Año y venue:** 1982 · IEEE Transactions on Information Theory, 28(2), 129–137  
**Nivel:** L2 · **Motor:** `kmeans`  
**Ficha completa:** [`P73_kmeans`](../../papers/foundational/P73_kmeans/README.md)

**Hito:** El algoritmo de agrupamiento más usado del mundo, con la demostración de que converge —y de que converge a un óptimo local, no al global.

- [doi:10.1109/TIT.1982.1056489](https://doi.org/10.1109/TIT.1982.1056489)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Resumir un conjunto de puntos con k representantes exige elegirlos minimizando el error cuadrático. El problema es combinatorio y su solución exacta, inabordable.
2. Ejecutar una implementación mínima de la propuesta: Alternar dos pasos que cada uno reduce el error: asignar cada punto a su representante más cercano, y recolocar cada representante en el centro de los puntos que le tocaron.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Steinhaus (1956), partición óptima
- P53


## 4. Intuición

Pon k banderas al azar. Cada punto se va con la bandera más cercana; cada bandera se mueve al centro de los que le tocaron. Repite. Se para siempre — y no siempre en el mismo sitio.


## 5. Concepto mínimo

```text
Repetir hasta que nada cambie:
    asignar : cada punto al centro más cercano       ← baja la inercia
    mover   : cada centro al promedio de los suyos   ← baja la inercia

Inercia = Σ ‖x − centro(x)‖².  Como baja en los dos pasos y hay un número
finito de asignaciones, el algoritmo TERMINA. En un óptimo LOCAL.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('kmeans', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántas iteraciones tardará en converger?
2. ¿Darán todos los arranques la misma inercia final?
3. ¿Qué le pasa a la inercia al aumentar k?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('kmeans', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('kmeans', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Converge en pocos pasos y la inercia nunca sube. Pero con ocho arranques aleatorios aparecen **inercias finales distintas**: una de 1,41 y otra de 61,59 sobre los mismos doce puntos. Converger no es encontrar el óptimo. Y la inercia decrece siempre al subir k.


## 10. Comentario pedagógico

Las dos consecuencias prácticas están en esa salida. Primera: hay que ejecutar varias veces y quedarse con la mejor —o usar k-means++ para inicializar—. Segunda: **no se puede elegir k minimizando la inercia**, porque el mínimo está en un grupo por punto. Hace falta otro criterio, y esa decisión no la toma el algoritmo.


## 11. Error o anti-patrón deliberado

Anti-patrón: elegir el número de grupos por la inercia.


In [ ]:
r = run_paper_lab('kmeans', seed=7)['result']
for fila in r['inercia_por_k']:
    print(f"k = {fila['k']:>2}  inercia = {fila['inercia']}")
print('Minimizar esta columna lleva a k = n. La inercia no elige k.')

## 12. Corrección

El criterio tiene que venir de fuera del algoritmo:


In [ ]:
print('opciones razonables: codo, silueta, criterio de informacion, o el dominio')
print('la mejor suele ser la ultima: cuantos grupos NECESITA quien va a usar esto')
r = run_paper_lab('kmeans', seed=7)['result']
print('inercias finales distintas con 8 arranques:', r['inercias_finales_distintas'])

## 13. Desafío guiado

Compara el mejor y el peor arranque en la salida y explica por qué el peor no es un error del algoritmo.


In [ ]:
r = run_paper_lab('kmeans', seed=3)['result']
show(r)

## 14. Desafío autónomo

Aplica k-medias a un conjunto real con variables en escalas distintas, primero sin estandarizar y después estandarizando. Documenta cuánto cambian los grupos y por qué.


## 15. Evidencia de aprendizaje

Guarda la tabla de inercias por arranque y por k, y tu criterio para elegir k.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P73_kmeans/README.md) · evaluación formal: [`assessments/papers/P73_kmeans.md`](../../assessments/papers/P73_kmeans.md)


## 16. Cierre

Ya se pueden agrupar puntos sin etiquetas. Con etiquetas, la pregunta cambia: qué pregunta hacer primero para separarlos.


## 17. Conexión con el siguiente hito

- P83

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
